# Entrega 2 - Integracao e Limpeza de Dados

**Projeto Final - Ciencia de Dados (UTFPR)**

Este notebook integra os dados de 2010 a 2024 usando:

- **IBGE - Tabela 5457**: area plantada, area colhida, quantidade produzida, rendimento medio e valor da producao, incluindo percentuais.
- **INMET**: medias mensais por UF.
- **Banco Central do Brasil - SGS**: IC-Br Agropecuaria, IC-Br Energia, IPCA monitorados, dolar comercial venda e IBC-Br.

A base integrada final e salva em `data/processed/dataset_integrado.csv` e `data/processed/dataset_final.parquet`. Antes do `dropna`, o notebook mostra a porcentagem de linhas com valores ausentes por coluna.

In [1]:
from pathlib import Path
from functools import reduce
import csv
import warnings

import numpy as np
import pandas as pd
import requests

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists() and PROJECT_ROOT.name == 'final':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data'
COMMODITY_DIR = DATA_DIR / 'commodity'
WEATHER_DIR = DATA_DIR / 'weather'
PROCESSED_DIR = DATA_DIR / 'processed'
FINAL_DIR = DATA_DIR / 'final'

ANO_INICIAL = 2010
ANO_FINAL = 2024
ANOS = list(range(ANO_INICIAL, ANO_FINAL + 1))
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

## 1. Clima mensal por UF

A base climatica processada e reutilizada quando ja existe e cobre o periodo 2010-2024. Caso contrario, o notebook recompila os arquivos brutos do INMET.

In [2]:
clima_processado_path = WEATHER_DIR / 'processed' / 'medias_mensais_por_estado_ano.csv'

if clima_processado_path.exists():
    df_clima = pd.read_csv(clima_processado_path)
    df_clima = df_clima[df_clima['ano'].between(ANO_INICIAL, ANO_FINAL)].copy()
    print(f'Clima carregado de {clima_processado_path.relative_to(PROJECT_ROOT)}: {df_clima.shape}')
else:
    pasta_raw = WEATHER_DIR / 'raw'
    df_estacoes = pd.read_csv(WEATHER_DIR / 'dados_estacao.csv')
    df_operante = df_estacoes[df_estacoes['situacao_status'] == 'Operante']
    registros_clima = []

    print(f'Processando {len(df_operante)} estacoes operantes...')
    for _, row in df_operante.iterrows():
        codigo = row['codigo_estacao']
        uf = row['estado_uf']
        arquivos_encontrados = list(pasta_raw.glob(f'dados_{codigo}_*.csv'))
        if not arquivos_encontrados:
            continue
        try:
            df_est = pd.read_csv(
                arquivos_encontrados[0],
                skiprows=10,
                sep=';',
                decimal=',',
                na_values=['null', '']
            )
            df_est = df_est.loc[:, ~df_est.columns.str.contains('^Unnamed')]
            df_est = df_est.dropna(how='all', axis=1)
            if df_est.empty or len(df_est.columns) < 7:
                continue

            df_est.columns = [
                'data_medicao', 'dias_precip', 'precip_total_mm',
                'pressao_media_mb', 'temp_media_c',
                'vento_max_ms', 'vento_media_ms'
            ]
            df_est['data_medicao'] = pd.to_datetime(df_est['data_medicao'], errors='coerce')
            df_est['ano'] = df_est['data_medicao'].dt.year
            df_est['mes'] = df_est['data_medicao'].dt.month
            df_est['uf'] = uf
            df_est['estacao'] = codigo

            num_cols = [
                'dias_precip', 'precip_total_mm', 'pressao_media_mb',
                'temp_media_c', 'vento_max_ms', 'vento_media_ms'
            ]
            for col in num_cols:
                df_est[col] = pd.to_numeric(df_est[col], errors='coerce')
                medias_mensais = df_est.groupby('mes')[col].transform('mean')
                df_est[col] = df_est[col].fillna(medias_mensais).fillna(df_est[col].mean()).fillna(0.0)

            registros_clima.append(df_est[['uf', 'ano', 'mes', 'estacao'] + num_cols])
        except Exception:
            continue

    df_clima_bruto = pd.concat(registros_clima, ignore_index=True)
    df_clima_bruto = df_clima_bruto[df_clima_bruto['ano'].between(ANO_INICIAL, ANO_FINAL)]
    df_clima = df_clima_bruto.groupby(['uf', 'ano', 'mes']).agg(
        precip_total_mm=('precip_total_mm', 'mean'),
        dias_precip=('dias_precip', 'mean'),
        temp_media_c=('temp_media_c', 'mean'),
        pressao_media_mb=('pressao_media_mb', 'mean'),
        vento_max_ms=('vento_max_ms', 'mean'),
        vento_media_ms=('vento_media_ms', 'mean'),
        n_estacoes=('estacao', 'nunique')
    ).reset_index()
    media_hist = df_clima.groupby('uf')['precip_total_mm'].transform('mean')
    std_hist = df_clima.groupby('uf')['precip_total_mm'].transform('std').replace(0, np.nan)
    df_clima['anomalia_precip'] = ((df_clima['precip_total_mm'] - media_hist) / std_hist).fillna(0.0)
    clima_processado_path.parent.mkdir(parents=True, exist_ok=True)
    df_clima.to_csv(clima_processado_path, index=False)

print(df_clima['ano'].min(), df_clima['ano'].max(), df_clima.shape)
df_clima.head()

Clima carregado de data\weather\processed\medias_mensais_por_estado_ano.csv: (4680, 11)
2010 2024 (4680, 11)


,uf,ano,mes,precip_total_mm,dias_precip,temp_media_c,pressao_media_mb,vento_max_ms,vento_media_ms,n_estacoes,anomalia_precip
0,AC,2010,1,272.8,26.0,25.50,987.650000,7.00,1.60,2,0.994857
1,AC,2010,2,207.0,19.5,25.95,987.809091,6.20,1.40,2,0.379624
2,AC,2010,3,188.8,25.0,25.85,988.254545,7.55,1.25,2,0.209453
3,AC,2010,4,102.6,14.5,26.05,989.125000,5.95,1.30,2,-0.596522
4,AC,2010,5,36.6,15.0,24.45,990.209091,8.90,1.30,2,-1.213626


## 2. Commodity - parser de blocos da Tabela 5457

O arquivo do IBGE contem varias tabelas empilhadas no mesmo CSV. A ingestao abaixo detecta cada linha `Variavel - ...`, reaproveita o cabecalho de ano/produto do bloco e junta as oito medidas solicitadas em uma unica base anual por UF, ano e cultura.

In [3]:
UF_MAP = {
    'Rondônia': 'RO', 'Rondonia': 'RO', 'Acre': 'AC', 'Amazonas': 'AM', 'Roraima': 'RR',
    'Pará': 'PA', 'Para': 'PA', 'Amapá': 'AP', 'Amapa': 'AP', 'Tocantins': 'TO',
    'Maranhão': 'MA', 'Maranhao': 'MA', 'Piauí': 'PI', 'Piaui': 'PI', 'Ceará': 'CE', 'Ceara': 'CE',
    'Rio Grande do Norte': 'RN', 'Paraíba': 'PB', 'Paraiba': 'PB', 'Pernambuco': 'PE', 'Alagoas': 'AL',
    'Sergipe': 'SE', 'Bahia': 'BA', 'Minas Gerais': 'MG', 'Espírito Santo': 'ES', 'Espirito Santo': 'ES',
    'Rio de Janeiro': 'RJ', 'São Paulo': 'SP', 'Sao Paulo': 'SP', 'Paraná': 'PR', 'Parana': 'PR',
    'Santa Catarina': 'SC', 'Rio Grande do Sul': 'RS', 'Mato Grosso do Sul': 'MS',
    'Mato Grosso': 'MT', 'Goiás': 'GO', 'Goias': 'GO', 'Distrito Federal': 'DF'
}

CULTURA_MAP = {
    'Cana-de-açúcar': 'Cana_de_acucar', 'Cana-de-acucar': 'Cana_de_acucar',
    'Mandioca': 'Mandioca',
    'Milho (em grão)': 'Milho', 'Milho (em grao)': 'Milho',
    'Soja (em grão)': 'Soja', 'Soja (em grao)': 'Soja'
}

VARIAVEIS_IBGE = {
    'Variável - Área plantada ou destinada à colheita (Hectares)': 'area_plantada_ha',
    'Variável - Área plantada ou destinada à colheita - percentual do total geral': 'area_plantada_percentual',
    'Variável - Área colhida (Hectares)': 'area_colhida_ha',
    'Variável - Área colhida - percentual do total geral': 'area_colhida_percentual',
    'Variável - Quantidade produzida (Toneladas)': 'quantidade_produzida_t',
    'Variável - Rendimento médio da produção (Quilogramas por Hectare)': 'rendimento_medio_producao_kg_ha',
    'Variável - Valor da produção (Mil Reais)': 'valor_producao_mil_reais',
    'Variável - Valor da produção - percentual do total geral': 'valor_producao_percentual',
}

TIPO_CULTURA = {
    'Cana_de_acucar': 'Industrial',
    'Mandioca': 'Raiz',
    'Milho': 'Grao',
    'Soja': 'Grao'
}

REGIAO = {
    'AC': 'Norte', 'AM': 'Norte', 'AP': 'Norte', 'PA': 'Norte', 'RO': 'Norte', 'RR': 'Norte', 'TO': 'Norte',
    'AL': 'Nordeste', 'BA': 'Nordeste', 'CE': 'Nordeste', 'MA': 'Nordeste', 'PB': 'Nordeste', 'PE': 'Nordeste',
    'PI': 'Nordeste', 'RN': 'Nordeste', 'SE': 'Nordeste',
    'DF': 'Centro-Oeste', 'GO': 'Centro-Oeste', 'MS': 'Centro-Oeste', 'MT': 'Centro-Oeste',
    'ES': 'Sudeste', 'MG': 'Sudeste', 'RJ': 'Sudeste', 'SP': 'Sudeste',
    'PR': 'Sul', 'RS': 'Sul', 'SC': 'Sul'
}

def parse_valor_ibge(valor):
    texto = str(valor).strip().strip('"')
    if texto in {'', '...', 'X'}:
        return np.nan
    if texto == '-':
        return 0.0
    return pd.to_numeric(texto.replace('.', '').replace(',', '.'), errors='coerce')

def parse_bloco_ibge(path, variavel, coluna_destino):
    with open(path, newline='', encoding='utf-8-sig') as f:
        rows = list(csv.reader(f, delimiter=';'))

    start = next(i for i, row in enumerate(rows) if row and row[0].strip() == variavel)
    year_idx = next(
        i for i in range(start + 1, len(rows))
        if rows[i] and rows[i][0] == 'Unidade da Federação' and any(c.strip().isdigit() for c in rows[i][1:])
    )
    product_idx = year_idx + 1
    year_row = rows[year_idx]
    product_row = rows[product_idx]

    anos_por_coluna = []
    ano_atual = None
    for cell in year_row[1:]:
        if cell.strip().isdigit():
            ano_atual = int(cell.strip())
        anos_por_coluna.append(ano_atual)

    registros = []
    for row in rows[product_idx + 1:]:
        if not row or row[0].startswith(('Fonte:', 'Tabela ', 'Variável -')):
            break
        uf = UF_MAP.get(row[0].strip())
        if uf is None:
            continue
        for pos, raw_value in enumerate(row[1:]):
            if pos >= len(anos_por_coluna) or pos + 1 >= len(product_row):
                continue
            ano = anos_por_coluna[pos]
            cultura = CULTURA_MAP.get(product_row[pos + 1].strip())
            if ano not in ANOS or cultura is None:
                continue
            registros.append({
                'uf': uf,
                'ano': ano,
                'cultura': cultura,
                coluna_destino: parse_valor_ibge(raw_value)
            })
    return pd.DataFrame(registros)

commodity_path = COMMODITY_DIR / 'yearly-production-per-state-historical.csv'
frames_commodity = [
    parse_bloco_ibge(commodity_path, variavel, coluna)
    for variavel, coluna in VARIAVEIS_IBGE.items()
]

df_agro = reduce(
    lambda left, right: left.merge(right, on=['uf', 'ano', 'cultura'], how='outer'),
    frames_commodity
)

df_agro['tipo_cultura'] = df_agro['cultura'].map(TIPO_CULTURA)
df_agro['regiao'] = df_agro['uf'].map(REGIAO)
df_agro = df_agro[df_agro['ano'].between(ANO_INICIAL, ANO_FINAL)].sort_values(['uf', 'ano', 'cultura']).reset_index(drop=True)

print(f'Commodity anual: {df_agro.shape[0]:,} linhas x {df_agro.shape[1]} colunas')
print(f'Anos: {df_agro["ano"].min()}-{df_agro["ano"].max()}')
df_agro.head()

Commodity anual: 1,539 linhas x 13 colunas
Anos: 2010-2024


,uf,ano,cultura,area_plantada_ha,area_plantada_percentual,area_colhida_ha,area_colhida_percentual,quantidade_produzida_t,rendimento_medio_producao_kg_ha,valor_producao_mil_reais,valor_producao_percentual,tipo_cultura,regiao
0,AC,2010,Cana_de_acucar,2769.0,2.15,1999.0,1.62,107251.0,53652.0,7209.0,1.78,Industrial,Norte
1,AC,2010,Mandioca,41108.0,31.91,40698.0,32.92,849667.0,20877.0,284211.0,70.27,Raiz,Norte
2,AC,2010,Milho,39784.0,30.88,39314.0,31.80,81125.0,2063.0,31533.0,7.80,Grao,Norte
3,AC,2010,Soja,100.0,0.08,100.0,0.08,330.0,3300.0,219.0,0.05,Grao,Norte
4,AC,2011,Cana_de_acucar,2654.0,2.01,2654.0,2.08,179044.0,67462.0,11185.0,2.77,Industrial,Norte


## 3. Dados economicos do SGS/BCB

A serie `1 - Dolar comercial venda` e diaria e o SGS limita consultas diarias a janelas menores. Por isso, todas as series sao buscadas em dois blocos: `2010-2018` e `2019-2024`, depois agregadas por media mensal.

In [4]:
SERIES_SGS = {
    27575: 'icbr_agropecuaria_brl',
    29041: 'icbr_agropecuaria_usd',
    27577: 'icbr_energia_brl',
    29039: 'icbr_energia_usd',
    4449: 'ipca_monitorados_total',
    1: 'dolar_comercial_venda',
    24363: 'ibc_br',
}

PERIODOS_SGS = [
    ('01/01/2010', '31/12/2018'),
    ('01/01/2019', '31/12/2024'),
]

def baixar_serie_sgs(codigo, nome_coluna):
    partes = []
    for data_inicial, data_final in PERIODOS_SGS:
        url = (
            f'https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados'
            f'?formato=json&dataInicial={data_inicial}&dataFinal={data_final}'
        )
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        payload = response.json()
        if isinstance(payload, dict):
            raise RuntimeError(f'Erro SGS {codigo}: {payload}')
        partes.append(pd.DataFrame(payload))

    df = pd.concat(partes, ignore_index=True).drop_duplicates()
    df['data'] = pd.to_datetime(df['data'], dayfirst=True, errors='coerce')
    df[nome_coluna] = pd.to_numeric(df['valor'].astype(str).str.replace(',', '.', regex=False), errors='coerce')
    df['ano'] = df['data'].dt.year
    df['mes'] = df['data'].dt.month
    df = df[df['ano'].between(ANO_INICIAL, ANO_FINAL)].dropna(subset=[nome_coluna])
    return df.groupby(['ano', 'mes'], as_index=False)[nome_coluna].mean()

frames_sgs = []
for codigo, nome_coluna in SERIES_SGS.items():
    serie = baixar_serie_sgs(codigo, nome_coluna)
    frames_sgs.append(serie)
    print(f'{codigo} - {nome_coluna}: {serie.shape[0]} meses')

df_sgs = reduce(lambda left, right: left.merge(right, on=['ano', 'mes'], how='outer'), frames_sgs)
df_sgs = df_sgs.sort_values(['ano', 'mes']).reset_index(drop=True)

print(f'SGS mensal: {df_sgs.shape[0]:,} linhas x {df_sgs.shape[1]} colunas')
df_sgs.head()

27575 - icbr_agropecuaria_brl: 180 meses


29041 - icbr_agropecuaria_usd: 180 meses


27577 - icbr_energia_brl: 180 meses


29039 - icbr_energia_usd: 180 meses


4449 - ipca_monitorados_total: 180 meses


1 - dolar_comercial_venda: 180 meses


24363 - ibc_br: 180 meses
SGS mensal: 180 linhas x 9 colunas


,ano,mes,icbr_agropecuaria_brl,icbr_agropecuaria_usd,icbr_energia_brl,icbr_energia_usd,ipca_monitorados_total,dolar_comercial_venda,ibc_br
0,2010,1,101.01,129.96,63.19,81.26,0.83,1.779820,87.94934
1,2010,2,102.85,127.54,62.53,77.46,0.42,1.841633,89.20680
2,2010,3,95.08,121.76,57.54,73.62,-0.14,1.785843,100.27383
3,2010,4,92.12,119.91,57.75,75.10,0.14,1.756570,95.67957
4,2010,5,92.38,116.54,58.99,74.37,0.33,1.813190,95.43544


## 4. Integracao final e remocao de NA

A tabela agricola e anual. Para integrar clima e SGS mensais, cada linha agricola e expandida para os 12 meses do ano antes dos merges. O percentual de linhas com NA e mostrado antes do `dropna`.

In [5]:
df_meses = pd.DataFrame({'mes': range(1, 13)})

df_integrado = (
    df_agro
    .merge(df_meses, how='cross')
    .merge(df_clima, on=['uf', 'ano', 'mes'], how='left')
    .merge(df_sgs, on=['ano', 'mes'], how='left')
)

df_integrado = df_integrado[df_integrado['ano'].between(ANO_INICIAL, ANO_FINAL)].copy()

linhas_antes = len(df_integrado)
percentual_linhas_com_na = df_integrado.isna().any(axis=1).mean() * 100
missing_por_coluna = (df_integrado.isna().mean() * 100).round(2)

print(f'Linhas antes do dropna: {linhas_antes:,}')
print(f'% de linhas com pelo menos um NA antes do dropna: {percentual_linhas_com_na:.2f}%')
print('\n% de NA por coluna:')
print(missing_por_coluna[missing_por_coluna > 0].sort_values(ascending=False))

df_integrado = df_integrado.dropna().reset_index(drop=True)

print(f'Linhas apos dropna: {len(df_integrado):,}')
print(f'NAs restantes: {int(df_integrado.isna().sum().sum())}')
print(f'Periodo final: {df_integrado["ano"].min()}-{df_integrado["ano"].max()}')

df_integrado.head()

Linhas antes do dropna: 18,468
% de linhas com pelo menos um NA antes do dropna: 3.70%

% de NA por coluna:
precip_total_mm     3.7
dias_precip         3.7
temp_media_c        3.7
pressao_media_mb    3.7
vento_max_ms        3.7
vento_media_ms      3.7
n_estacoes          3.7
anomalia_precip     3.7
dtype: float64
Linhas apos dropna: 17,784
NAs restantes: 0
Periodo final: 2010-2024


,uf,ano,cultura,area_plantada_ha,area_plantada_percentual,area_colhida_ha,area_colhida_percentual,quantidade_produzida_t,rendimento_medio_producao_kg_ha,valor_producao_mil_reais,...,vento_media_ms,n_estacoes,anomalia_precip,icbr_agropecuaria_brl,icbr_agropecuaria_usd,icbr_energia_brl,icbr_energia_usd,ipca_monitorados_total,dolar_comercial_venda,ibc_br
0,AC,2010,Cana_de_acucar,2769.0,2.15,1999.0,1.62,107251.0,53652.0,7209.0,...,1.60,2.0,0.994857,101.01,129.96,63.19,81.26,0.83,1.779820,87.94934
1,AC,2010,Cana_de_acucar,2769.0,2.15,1999.0,1.62,107251.0,53652.0,7209.0,...,1.40,2.0,0.379624,102.85,127.54,62.53,77.46,0.42,1.841633,89.20680
2,AC,2010,Cana_de_acucar,2769.0,2.15,1999.0,1.62,107251.0,53652.0,7209.0,...,1.25,2.0,0.209453,95.08,121.76,57.54,73.62,-0.14,1.785843,100.27383
3,AC,2010,Cana_de_acucar,2769.0,2.15,1999.0,1.62,107251.0,53652.0,7209.0,...,1.30,2.0,-0.596522,92.12,119.91,57.75,75.10,0.14,1.756570,95.67957
4,AC,2010,Cana_de_acucar,2769.0,2.15,1999.0,1.62,107251.0,53652.0,7209.0,...,1.30,2.0,-1.213626,92.38,116.54,58.99,74.37,0.33,1.813190,95.43544


In [6]:
csv_path = PROCESSED_DIR / 'dataset_integrado.csv'
parquet_path = PROCESSED_DIR / 'dataset_final.parquet'

df_integrado.to_csv(csv_path, index=False)
df_integrado.to_parquet(parquet_path, index=False)

print(f'CSV salvo em: {csv_path.relative_to(PROJECT_ROOT)}')
print(f'Parquet salvo em: {parquet_path.relative_to(PROJECT_ROOT)}')
print(df_integrado.shape)

CSV salvo em: data\processed\dataset_integrado.csv
Parquet salvo em: data\processed\dataset_final.parquet
(17784, 29)
